# UniFault (Foundation Model) on IMS — comparison baseline

**Goal:** fine-tune UniFault (fault-diagnosis foundation model, arxiv 2504.01373) on NASA IMS at risk levels **K=10/20/50 fault windows**, then measure **recall on a FIXED 8100-fault test set** so we can compare apples-to-apples against our `interp_align+amp` deliverable.

**Dataset:** NASA IMS (run-to-failure). Normal = first 70% of files, fault = last 10% of files (matches our harness zoning). Window = 1024 samples, single channel.

**Why this matters:** UniFault is a foundation model designed to *transfer across domains*. Even though we run it here on IMS, the question it answers (does a pretrained FM beat manifold-interpolation at extreme scarcity?) is methodology and carries over to the real DENSO dataset.

## 0. Setup — install deps (Kaggle has most pre-installed)

Kaggle already ships `pytorch-lightning`, `torchmetrics`, `timm`, `einops`, `pyarrow`. We only need to **clone the repo** and **download the Tiny pretrained checkpoint**.

In [ ]:
import os, glob, zipfile, subprocess
import torch, numpy as np
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

# clone UniFault
if not os.path.exists('UniFault'):
    subprocess.run('git clone --depth 1 https://github.com/emadeldeen24/UniFault.git', shell=True, check=True)
print('repo present:', os.path.isdir('UniFault'))

In [ ]:
import importlib
for m in ['pytorch_lightning','torchmetrics','timm','einops','pyarrow']:
    try:
        importlib.import_module(m); print(f'{m:>18} OK')
    except Exception as e:
        print(f'{m:>18} MISSING -> pip install '+m)
        subprocess.run('pip install -q '+m, shell=True)

## 1. Download Tiny pretrained model

Repo README links the Tiny checkpoint. It is a Dropbox ZIP containing `pretrain-epoch=1.ckpt`. Extract so that `lightning_logs/pretrained_model_dir/pretrain-epoch=1.ckpt` resolves. Our wrapper below will set `pretrain_model_dir` explicitly.

In [ ]:
TINY_URL = 'https://tinyurl.com/mtdytebs'  # redirects to Dropbox ZIP
if not os.path.exists('pretrained_models/Tiny'):
    subprocess.run(['curl','-sL',TINY_URL,'-o','/tmp/tiny.zip'], check=True)
    with zipfile.ZipFile('/tmp/tiny.zip') as z:
        z.extractall('/tmp/tiny_extract')
    os.makedirs('pretrained_models', exist_ok=True)
    # move the ckpt folder into pretrained_models/Tiny
    src = glob.glob('/tmp/tiny_extract/**', recursive=True)
    os.makedirs('pretrained_models/Tiny', exist_ok=True)
    for f in glob.glob('/tmp/tiny_extract/**/pretrain-epoch*.ckpt', recursive=True):
        import shutil; shutil.copy(f, 'pretrained_models/Tiny/'+os.path.basename(f))

ckpts = sorted(glob.glob('pretrained_models/Tiny/*.ckpt'))
print('ckpt found:', ckpts)
if ckpts:
    # derive the epoch id from filename (e.g. pretrain-epoch=1.ckpt -> 1)
    import re
    m = re.search(r'epoch=(\d+)', ckpts[0])
    EPOCH_ID = int(m.group(1)) if m else 1
    print('pretraining_epoch_id =', EPOCH_ID)

## 2. Build IMS dataset in UniFault parquet format

UniFault's `PHMDataset` reads `<data_path>/<data_id>/train.parquet|val.parquet|test.parquet`, each with columns:
- `samples`: list of `[num_channels, seq_len]`
- `labels`: 0 = normal (good), 1 = fault (bad)

We build windows directly from IMS files. **Zoning** (matches our harness):
- **normal** = first 70% of the per-test sorted file list
- **fault** = last 10% (90%→100%)

To solve the scarcity problem, the factory trains with only `K` fault windows in train (`--n-fault 10 20 50`) and keeps a FIXED large test fault set for a fair recall measurement.

`seq_len = 1024` (→ `num_patches = (1024-64)/64 + 1 = 16`, integer). Window = single channel of the DE channel (column 0).

In [ ]:
import numpy as np, glob, os
import pyarrow.parquet as pq, pyarrow as pa

# ---- IMS ROOT: SUA DUONG DAN O DAY NEU CAN (chua truc tiep test_1/test_2/test_3) ----
IMS_ROOT = '/kaggle/input/nasa_ims/NASA_IMS'   # <- chi sua dong nay
# --------------------------------------------------------------------------------
if not os.path.isdir(IMS_ROOT):
    raise SystemExit(f'Khong tim thay thu muc: {IMS_ROOT}\n'
                     f'Kiem tra bang: !ls /kaggle/input\n'
                     f'Ghi de len IMS_ROOT duong dan chua test_1/test_2/test_3')

NORMAL_END, FAULT_START = 0.70, 0.90
SEQ = 1024            # UniFault patch-friendly seq length
STRIDE = 512          # overlap for a bigger corpus
CHANNELS = 1          # use channel 0 (DE/vertical); UniFault unifies to univariate anyway

def ims_files(test_dir):
    import re
    pat = re.compile(r'^\d{4}\.\d{2}\.\d{2}\.\d{2}\.\d{2}\.\d{2}$')
    return sorted(p for p in glob.glob(test_dir+'/*') if os.path.isfile(p) and pat.match(os.path.basename(p)))

def load_channel(path, chan):
    a = np.genfromtxt(path, delimiter='\t')
    if a.size == 0: return None
    if a.ndim == 2:
        return a[:, min(chan, a.shape[1]-1)]   # pick the chosen channel (col 0)
    return a.ravel()

def windows(sig, seq, stride):
    if sig is None or len(sig) < seq: return np.empty((0, seq))
    n = (len(sig)-seq)//stride + 1
    idx = np.arange(n)[:,None]*stride + np.arange(seq)
    return sig[idx]

normal_wins, fault_wins = [], []
found_tests = 0
for test_dir in sorted(glob.glob(IMS_ROOT+'/test_*')):
    files = ims_files(test_dir)
    if len(files) < 3: continue
    found_tests += 1
    n = len(files)
    i_norm = int(n*NORMAL_END); i_fault = int(n*FAULT_START)
    f_norm = files[:i_norm]; f_fault = files[i_fault:]
    print(f'  test {os.path.basename(test_dir)}: {n} files (normal={len(f_norm)}, fault={len(f_fault)})')
    for p in f_norm:
        s = load_channel(p, CHANNELS)
        if s is not None:
            w = windows(s, SEQ, STRIDE)
            if len(w): normal_wins.append(w)
    for p in f_fault:
        s = load_channel(p, CHANNELS)
        if s is not None:
            w = windows(s, SEQ, STRIDE)
            if len(w): fault_wins.append(w)

X_norm = np.concatenate(normal_wins) if normal_wins else np.empty((0,SEQ))
X_fault = np.concatenate(fault_wins) if fault_wins else np.empty((0,SEQ))
print('=== RESULT ===')
print('tests found:', found_tests)
print('normal windows:', X_norm.shape, '| fault windows:', X_fault.shape)
assert found_tests > 0, 'KHONG test_* nao hop le (files < 3) - kiem tra duong dan ims.'
assert len(X_fault) > 0, 'KHONG tao duoc windows FAULT nao - kiem tra vung 90-100% khong co file.'
assert len(X_norm) > 0, 'KHONG tao duoc windows NORMAL nao.'


In [ ]:
import os
# ---- split, min-max normalize (fit on TRAIN only), write parquet ----
# We generate one dataset per scarcity level K to feed UniFault's few-shot path.
# UniFault's dataloader expects train/val/test .parquet under data_path/data_id/.

def minmax(X):
    lo, hi = X.min(), X.max()
    return (X - lo) / (hi - lo + 1e-9), lo, hi

def write_parquet(arr, labels, path):
    # UniFault's __getitem__ does x_data[index].squeeze(-1), so samples must be
    # [n, seq_len, num_channels] (channels LAST, =1 here) -> model gets [B, seq_len].
    # Storing nested python lists with explicit list_(list_(float32)) type avoids
    # pyarrow's "only 1-dim" error on 2D arrays.
    samples = arr[:, :, None].astype(np.float32).tolist()   # [n, seq_len, 1] -> list of lists
    t = pa.table({
        'samples': pa.array(samples, type=pa.list_(pa.list_(pa.float32()))),
        'labels': pa.array(labels, type=pa.int64()),
    })
    pq.write_table(t, path)

DATA_ROOT = '/kaggle/working'
def build_dataset(K, fault_frac_for_train=0.1):
    """Produce train/val/test parquet dir for scarcity level K.
    Val = split of train; test = the REST of normal + FULL fault pool.
    We keep a large 'test.fault' (≥8100) to mimic our fixed 8100-fault test.
    """
    dirpath = f'{DATA_ROOT}/unifault_data/ims_K{K}'
    os.makedirs(dirpath, exist_ok=True)

    # shuffle indices deterministically per K
    rng = np.random.default_rng(42 + K)  # seed+K mirrors our harness
    n_norm = len(X_norm)
    n_fault = len(X_fault)

    # train/test split for NORMAL: 70/30 (train keeps plenty normal)
    norm_idx = rng.permutation(n_norm)
    n_train_n = int(0.7*n_norm)
    n_train_n = min(n_train_n, 40000)  # cap duration
    X_train_n = X_norm[norm_idx[:n_train_n]]
    X_test_n  = X_norm[norm_idx[n_train_n:]]

    # fault: TRAIN keeps only K windows (scarcity); TEST keeps the rest (big)
    n_fault_avail = len(X_fault)
    if K >= n_fault_avail:
        raise SystemExit(f'K={K} nhuong hon/so bang so fault windows co ({n_fault_avail}). Giam K hoac them du lieu.')
    fault_idx = rng.permutation(n_fault_avail)
    X_train_f = X_fault[fault_idx[:K]]
    X_test_f  = X_fault[fault_idx[K:]]

    # min-max normalize on TRAIN only
    X_train_n, lo, hi = minmax(X_train_n)
    X_train_f = (X_train_f - lo)/(hi - lo + 1e-9)
    X_test_n  = (X_test_n  - lo)/(hi - lo + 1e-9)
    X_test_f  = (X_test_f  - lo)/(hi - lo + 1e-9)

    # val = 10% of train
    nval_n = int(0.1*len(X_train_n)); nval_f = int(0.1*len(X_train_f))
    X_val_n, X_val_f = X_train_n[:nval_n], X_train_f[:nval_f]
    X_train_n = X_train_n[nval_n:]; X_train_f = X_train_f[nval_f:]

    # UniFault keeps train as ONE parquet with both classes (labels carry the class):
    train_samples = np.concatenate([X_train_n, X_train_f]); train_labels = np.concatenate([np.zeros(len(X_train_n)), np.ones(len(X_train_f))]).astype(np.int64)
    write_parquet(train_samples, train_labels, f'{dirpath}/train.parquet')
    val_samples = np.concatenate([X_val_n, X_val_f]); val_labels = np.concatenate([np.zeros(len(X_val_n)), np.ones(len(X_val_f))]).astype(np.int64)
    write_parquet(val_samples, val_labels, f'{dirpath}/val.parquet')
    test_samples = np.concatenate([X_test_n, X_test_f]); test_labels = np.concatenate([np.zeros(len(X_test_n)), np.ones(len(X_test_f))]).astype(np.int64)
    write_parquet(test_samples, test_labels, f'{dirpath}/test.parquet')
    print(f'K={K}: train {train_samples.shape} (fault={len(X_train_f)}), val {val_samples.shape}, test {test_samples.shape} (fault={len(X_test_f)})')

for K in [10, 20, 50]:
    build_dataset(K)
print('datasets built under', DATA_ROOT+'/unifault_data')


In [ ]:
# HOTFIX: upstream utils.py uses importlib.util + ast without importing them -> NameError at runtime.
# Patch it so fine_tune.py (which calls save_copy_of_files) does not crash.
utils_p = 'UniFault/utils.py'
src = open(utils_p).read()
patch = 'import importlib\nimport ast\n'
if 'import importlib' not in src:
    # insert after the sklearn import line
    src = src.replace('from sklearn.metrics import classification_report, accuracy_score\n',
                      'from sklearn.metrics import classification_report, accuracy_score\n' + patch)
    open(utils_p, 'w').write(src)
    print('patched utils.py (added importlib + ast)')
else:
    print('utils.py already patched')

## 3. Run UniFault fine-tune per K

We call `fine_tune.py` for each `K` and `data_id=ims_K{K}`, loading the pretrained Tiny weights. This trains the classification head (and fine-tunes) with only `K+warmup normal` fault labels — the few-shot core of the comparison.

We **freeze the backbone** (or close to it) via the repo's default adapter path. The repo uses `--load_from_pretrained True --pretraining_epoch_id <id>`.

In [ ]:
import os, subprocess, sys
sys.path.insert(0, 'UniFault')

# The pretrained_model_dir that fine_tune.py's apply_model_config sets is relative.
# We place our ckpt so the default 'pretrained_models/Tiny/pretrain-epoch=1.ckpt' resolves.
os.makedirs('pretrained_models/Tiny', exist_ok=True)

RESULTS = {}
def run_finetune(K, epochs=30):
    data_id = f'ims_K{K}'
    os.makedirs('checkpoints', exist_ok=True)
    cmd = ["python", "UniFault/fine_tune.py",
           "--data_path", f"{DATA_ROOT}/unifault_data",
           "--data_id", data_id,
           "--data_percentage", "100",
           "--model_type", "tiny",
           "--model_id", f"ims_K{K}",
           "--gpu_id", "0",
           "--num_epochs", str(epochs),
           "--patience", "20",
           "--batch_size", "64",
           "--lr", "3e-4",
           "--load_from_pretrained", "True",
           "--pretraining_epoch_id", "1",
           "--random_seed", "42"]
    print(' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, cwd='/kaggle/working')
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1500:])
    return r.returncode

for K in [10, 20, 50]:
    run_finetune(K)

## 4. Extract Test Recall (our metric)

UniFault reports **accuracy/F1** on ITS OWN test split. That is NOT our metric (it is vulnerable to imbalance, and our baseline is recall on a **fixed 8100-fault test**). To get apples-to-apples recall, we:
- load the `best.ckpt`, run inference on a fixed test fault pool, and threshold on probability to read recall.

We reuse our own `X_test_f` from step 2 as the fixed fault set and report recall at the default 0.5 threshold.

In [ ]:
import torch, glob, re
from model.model import Transformer_bkbone

BEST = sorted(glob.glob('checkpoints/**/best.ckpt', recursive=True))
print('best.ckpt files:', BEST)

def mean_std_baseline():
    # placeholder: compares against our interp_align+amp+k_mix6 = 0.766/0.748/0.788
    return {'10': 0.766, '20': 0.748, '50': 0.788}

# Build a minimal test loader from the K-specific parquet for a fair measure
def load_metadata(data_id):
    t = pq.read_table(f'{DATA_ROOT}/unifault_data/{data_id}/test.parquet')
    arrays = np.array(t['samples'].to_pylist())
    labels = t['labels'].to_numpy()
    return arrays[:,0,:], labels

print(mean_std_baseline())

## Summary

After the runs, the comparison table is:

| K | Baseline (no augment) | interp_align+amp+k_mix6 | UniFault (FM) |
|---|---|---|---|
| 10 | 0.252 | 0.766 | `<fill>` |
| 20 | 0.383 | 0.748 | `<fill>` |
| 50 | 0.490 | 0.788 | `<fill>` |

**Important caveat:** UniFault is a *foundation* model — it is designed to transfer across domains, so its answer here (does an FM beat manifold-interpolation at extreme scarcity?) also applies to the unseen real data. It still needs to be measured, not assumed.